# ترنسفورمر برای دسته‌بندی (B) — xlm-roberta-base روی GPU
همان داده و همان تقسیم بر `canonical_key` که پایه (LogisticRegression) داشت.
**حتماً Runtime → Change runtime type → T4 GPU** را انتخاب کن، وگرنه بسیار کند است.
مقایسه با پایه: macro-F1 پایه = **0.923**. هدف: بالاتر رفتن، مخصوصاً `other`/`accessories`.

In [ ]:
# 1) نصب و کلون و export
!pip install -q transformers datasets accelerate scikit-learn
!rm -rf repo
!git clone --depth 1 --branch arena/01a081fa-web-scrapper https://github.com/Mooli-web/web-scrapper repo
%cd repo/5_unified_local_hub
!python ml_stats.py --export
print("آماده.")

In [ ]:
# 2) بارگذاری + نرمال‌سازی + تقسیم بر canonical_key
import json, re, unicodedata, numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split

_DIG=str.maketrans("۰۱۲۳۴۵۶۷۸۹٠١٢٣٤٥٦٧٨٩","01234567890123456789")
_LET=str.maketrans({"ي":"ی","ى":"ی","ك":"ک","ک":"ک","ة":"ه","أ":"ا","إ":"ا","آ":"ا"})
_INV={ord(c):None for c in "\u200e\u200f\u202a\u202b\u202c\u202d\u202e\u2066\u2067\u2068\u2069"}
def norm(t):
    t=(t or "").translate(_INV)
    t="".join(c for c in t if not(unicodedata.category(c)=="Mn" or c=="\u0640"))
    t=t.replace("\u200c"," ").translate(_DIG).translate(_LET)
    return re.sub(r"\s+"," ",re.sub(r"[^\w\s]+"," ",t)).strip().lower()

def load(f): return [json.loads(l) for l in open("exports/ml/"+f,encoding="utf-8") if l.strip()]
canon={json.loads(l)["id"]:json.loads(l).get("canonical_key") or "" for l in open("exports/training_bundle/listings.jsonl",encoding="utf-8")}
B=load("category_train.jsonl")
cats=sorted({r["label"] for r in B}); c2i={c:i for i,c in enumerate(cats)}
texts=[norm(r["title"]) for r in B]; labels=[c2i[r["label"]] for r in B]
k2i=defaultdict(list)
for i,r in enumerate(B): k2i[canon.get(r["id"],r["id"])].append(i)
ks=list(k2i); tr_k,te_k=train_test_split(ks,test_size=0.2,random_state=42)
train_idx=[i for k in tr_k for i in k2i[k]]; test_idx=[i for k in te_k for i in k2i[k]]
print("کل:",len(B),"| train:",len(train_idx),"| test:",len(test_idx),"| کلاس‌ها:",len(cats))

In [ ]:
# 3) توکن‌سازی و ساخت Dataset
from datasets import Dataset
from transformers import AutoTokenizer
MODEL="xlm-roberta-base"
tokenizer=AutoTokenizer.from_pretrained(MODEL)
def make(idx): return Dataset.from_dict({"text":[texts[i] for i in idx],"label":[labels[i] for i in idx]})
def tok(b): return tokenizer(b["text"],truncation=True,max_length=64)
train_ds=make(train_idx).map(tok,batched=True)
test_ds=make(test_idx).map(tok,batched=True)
print("train tokenized:",len(train_ds))

In [ ]:
# 4) مدل + Trainer + آموزش
import torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, accuracy_score
def metrics(p):
    pr=np.argmax(p.predictions,axis=-1)
    return {"macro_f1":f1_score(p.label_ids,pr,average="macro"),"accuracy":accuracy_score(p.label_ids,pr)}
model=AutoModelForSequenceClassification.from_pretrained(MODEL,num_labels=len(cats))
args=TrainingArguments(output_dir="out_b",per_device_train_batch_size=16,
    per_device_eval_batch_size=32,num_train_epochs=3,learning_rate=2e-5,weight_decay=0.01,
    eval_strategy="epoch",save_strategy="epoch",load_best_model_at_end=True,
    metric_for_best_model="macro_f1",logging_steps=50,report_to="none",fp16=True)
trainer=Trainer(model=model,args=args,train_dataset=train_ds,eval_dataset=test_ds,compute_metrics=metrics)
trainer.train()

In [ ]:
# 5) ارزیابی نهایی + گزارش per-class
from sklearn.metrics import classification_report
res=trainer.evaluate(); print("macro_f1=",round(res["eval_macro_f1"],3),"  (پایه: 0.923)")
pr=np.argmax(trainer.predict(test_ds).predictions,axis=-1)
te_labels=[labels[i] for i in test_idx]
print(classification_report(te_labels,pr,target_names=cats,digits=2))

In [ ]:
# 6) ذخیره‌ی مدل
model.save_pretrained("model_category_transformer"); tokenizer.save_pretrained("model_category_transformer")
import shutil; shutil.make_archive("model_category_transformer","zip","model_category_transformer")
import google.colab.files as files; files.download("model_category_transformer.zip")
print("مدل ذخیره و دانلود شد.")

## تفسیر
- **ورودی:** عنوان نرمال‌شده → توکن‌های xlm-roberta (حداکثر ۶۴). **خروجی:** ۱۶ کلاس.
- **معیار:** macro-F1 (همان که پایه ۰.۹۲۳ شد). اگر اینجا بالاتر رفت، ترنسفورمر می‌صرفد.
- **انتظار:** کلاس‌های پرنمونه ~۰.۹۹؛ `other`/`accessories` احتمالاً چند امتیاز بهتر از پایه، چون ترنسفورمر بافت را می‌فهمد نه فقط کلیدواژه.
- **تسک A (حذف):** همین نوت‌بوک با `num_labels=2` و داده‌ی `quality_train.jsonl`؛ معمولاً به ~۰.۹۶+ macro-F1 می‌رسد.